# 04 — Fine-Tuned Demucs: Guitar Stem Ablation

**Hypothesis to test:**  
The pre-trained htdemucs fails on guitar duets primarily because its stem
vocabulary (vocals / drums / bass / other) has no concept of guitar1 vs guitar2.
The entire guitar signal collapses into `other`, giving the model no gradient
signal to separate the two guitars.  

**Experiment:**  
Replace only the output head (4 stems → 2 guitar stems) and fine-tune in two phases:
1. **Head-only warm-up** — encoder frozen, only the new projection layer trained  
2. **Full fine-tune** — all weights unfrozen at a much smaller learning rate  

**Expected result:**  
- If fine-tuning significantly outperforms blind htdemucs → missing stem identity was the primary failure mode  
- If performance remains poor despite knowing the task → timbral similarity is the fundamental bottleneck, and score-informed conditioning (notebook 05) is necessary  

**Ablation ladder:**
```
Blind htdemucs (NB 03)  →  Fine-tuned Demucs (NB 04)  →  Score-informed Demucs (NB 05)
```

In [1]:
import sys
sys.path.insert(0, "..")

import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import IPython.display as ipd
from pathlib import Path

from src.data_processing import GuitarDuetsLoader, load_audio, DEMUCS_SR
from src.baseline_model import compute_sdr, compute_si_sdr
from src.finetune_demucs import (
    DemucsGuitarFineTuner, FinetuneConfig, build_guitar_demucs,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

DATASET_ROOT = "../dataset"
OUTPUT_DIR   = Path("../outputs/finetune")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 1. Dataset Split

In [2]:
loader = GuitarDuetsLoader(DATASET_ROOT)
print(loader.summary())

train_tracks = loader.real_tracks + loader.synth_tracks
val_tracks   = loader.real_test_tracks

print(f"\nTrain tracks : {len(train_tracks)}")
print(f"Val   tracks : {len(val_tracks)}")

{'total_tracks': 70, 'subsets': {'real': {'count': 26, 'has_midi': 0}, 'real_test': {'count': 8, 'has_midi': 0}, 'synth': {'count': 36, 'has_midi': 36}}}

Train tracks : 62
Val   tracks : 8


## 2. Inspect the Model Surgery

Sanity-check that the stem count was correctly replaced before any training.

In [3]:
model = build_guitar_demucs(model_name="htdemucs", device=device, freeze_encoder=True)

print("New stem names :", model.sources)
print()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)")
print()

# Identify which layers are being trained (head-only mode)
print("Trainable layers (head-only warm-up):")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {name}  {tuple(p.shape)}")

del model  # release memory before training

INFO src.finetune_demucs: Loading base model 'htdemucs'...
INFO src.finetune_demucs: Stems (unchanged): ['drums', 'bass', 'other', 'vocals']
INFO src.finetune_demucs: Mapping: vocals[3]→guitar1, other[2]→guitar2
INFO src.finetune_demucs: Encoder frozen (2849112 params). Decoder trainable (39135344 params).


New stem names : ['drums', 'bass', 'other', 'vocals']

Total params    : 41,984,456
Trainable params: 39,135,344  (93.2%)

Trainable layers (head-only warm-up):
  decoder.0.conv_tr.weight  (384, 192, 8, 1)
  decoder.0.conv_tr.bias  (192,)
  decoder.0.rewrite.weight  (768, 384, 3, 3)
  decoder.0.rewrite.bias  (768,)
  decoder.0.dconv.layers.0.0.weight  (48, 384, 3)
  decoder.0.dconv.layers.0.0.bias  (48,)
  decoder.0.dconv.layers.0.1.weight  (48,)
  decoder.0.dconv.layers.0.1.bias  (48,)
  decoder.0.dconv.layers.0.3.weight  (768, 48, 1)
  decoder.0.dconv.layers.0.3.bias  (768,)
  decoder.0.dconv.layers.0.4.weight  (768,)
  decoder.0.dconv.layers.0.4.bias  (768,)
  decoder.0.dconv.layers.0.6.scale  (384,)
  decoder.0.dconv.layers.1.0.weight  (48, 384, 3)
  decoder.0.dconv.layers.1.0.bias  (48,)
  decoder.0.dconv.layers.1.1.weight  (48,)
  decoder.0.dconv.layers.1.1.bias  (48,)
  decoder.0.dconv.layers.1.3.weight  (768, 48, 1)
  decoder.0.dconv.layers.1.3.bias  (768,)
  decoder.0.dconv.la

## 3. Configure and Run Fine-Tuning

In [4]:
SEGMENT_DIR = "../outputs/segments"

cfg = FinetuneConfig(
    output_dir              = str(OUTPUT_DIR),
    segment_dir             = SEGMENT_DIR,
    model_name              = "htdemucs",
    device                  = device,
    segment_duration        = 2.0,
    hop_duration            = 1.0,
    epochs_head_only        = 10,
    epochs_full             = 40,
    batch_size              = 4,
    grad_accumulation_steps = 4,
    use_mixed_precision     = True,
    lr_head                 = 1e-3,
    lr_full                 = 3e-5,
    weight_l1               = 1.0,
    weight_stft             = 1.0,
    weight_si_sdr           = 0.5,
    val_every_n_epochs      = 5,
    early_stop_patience     = 15,
)

trainer = DemucsGuitarFineTuner(cfg)
trainer.setup_model(freeze_encoder=True)

trainer.train(
    segment_dir    = SEGMENT_DIR,
    train_subsets  = ["real", "synth"],
    val_subsets    = ["real_test"],
)

INFO src.finetune_demucs: Loading base model 'htdemucs'...
INFO src.finetune_demucs: Stems (unchanged): ['drums', 'bass', 'other', 'vocals']
INFO src.finetune_demucs: Mapping: vocals[3]→guitar1, other[2]→guitar2
INFO src.finetune_demucs: Encoder frozen (2849112 params). Decoder trainable (39135344 params).
INFO src.finetune_demucs: Loading training segments from ../outputs/segments ...
INFO src.finetune_demucs:   real: 811 segments
INFO src.finetune_demucs:   synth: 1681 segments
INFO src.finetune_demucs: Dataset ready: 2492 segments total
INFO src.finetune_demucs: Loading validation segments...
INFO src.finetune_demucs:   real_test: 82 segments
INFO src.finetune_demucs: Dataset ready: 82 segments total
INFO src.finetune_demucs: ============================================================
INFO src.finetune_demucs: PHASE 1: Head-only warm-up (10 epochs)
INFO src.finetune_demucs: ============================================================


Phase 1:   0%|          | 0/10 [00:00<?, ?epoch/s]

  Epoch 1:   0%|          | 0/623 [00:00<?, ?batch/s]

KeyboardInterrupt: 

## 4. Training Curves

In [ ]:
import json

log_path = OUTPUT_DIR / "training_log.json"
with open(log_path) as f:
    log = json.load(f)

df_log = pd.DataFrame(log)
df_log["global_epoch"] = range(1, len(df_log) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
for phase, grp in df_log.groupby("phase"):
    axes[0].plot(grp["global_epoch"], grp["train_loss"], label=f"train (phase {phase})")
    if "val_loss" in grp.columns:
        axes[0].plot(grp["global_epoch"], grp["val_loss"].dropna(),
                     linestyle="--", label=f"val (phase {phase})")

# Add phase boundary
phase1_end = df_log[df_log["phase"] == 1]["global_epoch"].max()
axes[0].axvline(phase1_end, color="gray", ls=":", alpha=0.7, label="phase boundary")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].legend(fontsize=8)

# SI-SDR validation
val_rows = df_log.dropna(subset=["val_si_sdr"])
if not val_rows.empty:
    axes[1].plot(val_rows["global_epoch"], val_rows["val_si_sdr"], marker="o", color="steelblue")
    axes[1].axhline(0, color="red", ls="--", alpha=0.5, label="0 dB baseline")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("SI-SDR (dB)")
    axes[1].set_title("Validation SI-SDR")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No validation data", ha="center", va="center",
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

## 5. Evaluate on Test Set

Compare fine-tuned model against the blind Demucs baseline from notebook 03.

In [ ]:
from tqdm.notebook import tqdm

eval_tracks = val_tracks if val_tracks else loader.synth_tracks[:5]
best_ckpt   = str(OUTPUT_DIR / "best_model.pt")

all_results = []
for track in tqdm(eval_tracks, desc="Evaluating fine-tuned model"):
    try:
        estimates = trainer.separate(track.mix_path, checkpoint=best_ckpt)
        g1 = load_audio(track.guitar1_path, sr=DEMUCS_SR, mono=True)
        g2 = load_audio(track.guitar2_path, sr=DEMUCS_SR, mono=True)

        pred1 = estimates["guitar1"].mean(axis=0)
        pred2 = estimates["guitar2"].mean(axis=0)

        min_len = min(len(g1.waveform), len(g2.waveform), len(pred1))
        g1w, g2w = g1.waveform[:min_len], g2.waveform[:min_len]
        p1, p2   = pred1[:min_len], pred2[:min_len]

        # Permutation-invariant evaluation
        sdr_a = (compute_sdr(g1w, p1) + compute_sdr(g2w, p2)) / 2
        sdr_b = (compute_sdr(g1w, p2) + compute_sdr(g2w, p1)) / 2
        best_sdr = max(sdr_a, sdr_b)

        if sdr_a >= sdr_b:
            si1 = compute_si_sdr(g1w, p1)
            si2 = compute_si_sdr(g2w, p2)
        else:
            si1 = compute_si_sdr(g1w, p2)
            si2 = compute_si_sdr(g2w, p1)

        all_results.append({
            "track_id"  : track.track_id,
            "piece_name": track.piece_name,
            "SDR_G1"    : si1,
            "SDR_G2"    : si2,
            "avg_SDR"   : best_sdr,
            "SI-SDR_G1" : si1,
            "SI-SDR_G2" : si2,
            "avg_SI-SDR": (si1 + si2) / 2,
        })
    except Exception as e:
        print(f"Error on {track.track_id}: {e}")

df_ft = pd.DataFrame(all_results)
print(df_ft[["track_id", "avg_SDR", "avg_SI-SDR"]].round(2).to_string(index=False))

## 6. Side-by-Side Comparison: Baseline vs Fine-tuned

Paste in the baseline `avg_SDR` values from notebook 03 for a direct comparison.

In [ ]:
# ── Load baseline results (saved from NB 03) ──────────────────────────────
# If you saved df_results from NB 03:
#   df_baseline = pd.read_csv("../outputs/baseline/results.csv")
# Otherwise paste mean values directly:
BASELINE_MEAN_SDR = None   # e.g. -3.2  ← fill in from NB 03
FINETUNE_MEAN_SDR = float(df_ft["avg_SDR"].mean())

print("=" * 50)
print("ABLATION SUMMARY")
print("=" * 50)
if BASELINE_MEAN_SDR is not None:
    delta = FINETUNE_MEAN_SDR - BASELINE_MEAN_SDR
    print(f"Blind htdemucs   avg SDR : {BASELINE_MEAN_SDR:.2f} dB")
    print(f"Fine-tuned Demucs avg SDR: {FINETUNE_MEAN_SDR:.2f} dB")
    print(f"Δ SDR                    : {delta:+.2f} dB")
    print()
    if delta > 3.0:
        print("→ Large gain: stem-identity was the primary failure mode.")
        print("  Score conditioning (NB 05) should add further improvement.")
    elif delta > 0.5:
        print("→ Moderate gain: both stem-identity and timbral similarity contribute.")
        print("  Score conditioning is still expected to help.")
    else:
        print("→ Minimal gain: timbral similarity is the dominant problem.")
        print("  Score conditioning (NB 05) is essential, not optional.")
else:
    print(f"Fine-tuned Demucs avg SDR: {FINETUNE_MEAN_SDR:.2f} dB")
    print("(Set BASELINE_MEAN_SDR from NB 03 to see the delta)")

## 7. Listen to Fine-Tuned Outputs

In [ ]:
demo_track = eval_tracks[0]
estimates  = trainer.separate(demo_track.mix_path, checkpoint=best_ckpt)

mix   = load_audio(demo_track.mix_path,    sr=DEMUCS_SR, mono=True)
gt_g1 = load_audio(demo_track.guitar1_path, sr=DEMUCS_SR, mono=True)
gt_g2 = load_audio(demo_track.guitar2_path, sr=DEMUCS_SR, mono=True)

VIS_DUR = min(10, mix.duration)
vis_n   = int(VIS_DUR * DEMUCS_SR)

print("=== Input Mixture ===")
ipd.display(ipd.Audio(mix.waveform[:vis_n], rate=DEMUCS_SR))

print("\n=== Ground Truth: Guitar 1 ===")
ipd.display(ipd.Audio(gt_g1.waveform[:vis_n], rate=DEMUCS_SR))

print("\n=== Fine-tuned Estimate: Guitar 1 ===")
ipd.display(ipd.Audio(estimates["guitar1"].mean(axis=0)[:vis_n], rate=DEMUCS_SR))

print("\n=== Ground Truth: Guitar 2 ===")
ipd.display(ipd.Audio(gt_g2.waveform[:vis_n], rate=DEMUCS_SR))

print("\n=== Fine-tuned Estimate: Guitar 2 ===")
ipd.display(ipd.Audio(estimates["guitar2"].mean(axis=0)[:vis_n], rate=DEMUCS_SR))